In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

gemma_4b_five = pd.read_csv('slm_results/five_results/five_results_unsloth_gemma-3-4b-it-bnb-4bit.csv')
gemma_12b_five = pd.read_csv('slm_results/five_results/five_results_unsloth_gemma-3-12b-it-bnb-4bit.csv')
granite_8b_five = pd.read_csv('slm_results/five_results/five_results_unsloth_granite-3.2-8b-instruct-bnb-4bit.csv')
meta_8b_five = pd.read_csv('slm_results/five_results/five_results_unsloth_Meta-Llama-3.1-8B-Instruct-bnb-4bit.csv')
mistral_7b_five = pd.read_csv('slm_results/five_results/five_results_unsloth_mistral-7b-instruct-v0.3-bnb-4bit.csv')
qwen_7b_five = pd.read_csv('slm_results/five_results/five_results_unsloth_Qwen2.5-7B-Instruct-bnb-4bit.csv')

gemma_4b_single = pd.read_csv('slm_results/single_results/single_results_unsloth_gemma-3-4b-it-bnb-4bit.csv')
gemma_12b_single = pd.read_csv('slm_results/single_results/single_results_unsloth_gemma-3-12b-it-bnb-4bit.csv')
granite_8b_single = pd.read_csv('slm_results/single_results/single_results_unsloth_granite-3.2-8b-instruct-bnb-4bit.csv')
meta_8b_single = pd.read_csv('slm_results/single_results/single_results_unsloth_Meta-Llama-3.1-8B-Instruct-bnb-4bit.csv')
mistral_7b_single = pd.read_csv('slm_results/single_results/single_results_unsloth_mistral-7b-instruct-v0.3-bnb-4bit.csv')
qwen_7b_single = pd.read_csv('slm_results/single_results/single_results_unsloth_Qwen2.5-7B-Instruct-bnb-4bit.csv')


In [ ]:
# Make a graph of the average accuracy of each model on the five results and the average latency of each model on the five results 
models = ['Gemma 4B', 'Gemma 12B', 'Granite 8B', 'Meta 8B', 'Mistral 7B', 'Qwen 7B']
accuracies = [
    gemma_4b_five['accuracy'].mean(),
    gemma_12b_five['accuracy'].mean(),
    granite_8b_five['accuracy'].mean(),
    meta_8b_five['accuracy'].mean(),
    mistral_7b_five['accuracy'].mean(),
    qwen_7b_five['accuracy'].mean()
]
latencies = [
    gemma_4b_five['avg_time'].mean(),
    gemma_12b_five['avg_time'].mean(),
    granite_8b_five['avg_time'].mean(),
    meta_8b_five['avg_time'].mean(),
    mistral_7b_five['avg_time'].mean(),
    qwen_7b_five['avg_time'].mean()
]
# Make a scatter plot of the average accuracy vs average latency of each model
plt.figure(figsize=(10, 6))
sns.scatterplot(x=latencies, y=accuracies, hue=models, s=100)
plt.title('Average Accuracy vs Average Latency of Models on Five Results')
plt.xlabel('Average Latency (s)')
plt.ylabel('Average Accuracy (%)')
plt.legend(title='Models')
plt.grid()
plt.show()

In [ ]:
# Compare category accuracy (14 categories) across the 6 models in five_results
model_frames = {
    'Gemma 4B': gemma_4b_five,
    'Gemma 12B': gemma_12b_five,
    'Granite 8B': granite_8b_five,
    'Meta 8B': meta_8b_five,
    'Mistral 7B': mistral_7b_five,
    'Qwen 7B': qwen_7b_five,
}

category_accuracy = pd.DataFrame({
    model: df.groupby('category')['accuracy'].mean()
    for model, df in model_frames.items()
}).sort_index()

print('Mean accuracy by category (fraction):')
display(category_accuracy.round(4))

overall_mean = category_accuracy.mean(axis=0).sort_values(ascending=False)
print('Overall mean accuracy by model (fraction):')
print(overall_mean.round(4))

plt.figure(figsize=(13, 6))
sns.heatmap(
    category_accuracy * 100,
    annot=True,
    fmt='.1f',
    cmap='Blues',
    vmin=0,
    vmax=100,
    cbar_kws={'label': 'Accuracy (%)'}
)
plt.title('Five Results: Accuracy by Category and Model (%)')
plt.xlabel('Model')
plt.ylabel('Category')
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()



In [ ]:
# 80% vote on single_results using two 5-model variants
# complex = 0 if >=4/5 models are correct, else 1

def build_vote_df(model_frames, version_name):
    base = next(iter(model_frames.values()))
    out = base[['prompt', 'category', 'correct_answer']].copy()

    correctness_cols = []
    for model_name, df in model_frames.items():
        pred = df['model_answer'].astype(str).str.strip().str.upper()
        truth = df['correct_answer'].astype(str).str.strip().str.upper()
        col = f"{model_name}_correct"
        out[col] = (pred == truth).astype(int)
        correctness_cols.append(col)

    out['correct_models'] = out[correctness_cols].sum(axis=1)
    out['vote_accuracy'] = out['correct_models'] / 5.0
    out['complex'] = (out['correct_models'] <= 3).astype(int)
    out['version'] = version_name
    return out

models_with_gemma_4b = {
    'Gemma 4B': gemma_4b_single,
    'Granite 8B': granite_8b_single,
    'Meta 8B': meta_8b_single,
    'Mistral 7B': mistral_7b_single,
    'Qwen 7B': qwen_7b_single,
}

models_with_gemma_12b = {
    'Gemma 12B': gemma_12b_single,
    'Granite 8B': granite_8b_single,
    'Meta 8B': meta_8b_single,
    'Mistral 7B': mistral_7b_single,
    'Qwen 7B': qwen_7b_single,
}

vote_df_with_gemma_4b = build_vote_df(models_with_gemma_4b, 'with_gemma_4b')
vote_df_with_gemma_12b = build_vote_df(models_with_gemma_12b, 'with_gemma_12b')

print('DataFrame with Gemma 4B included:')
display(vote_df_with_gemma_4b.head())
print('DataFrame with Gemma 12B included:')
display(vote_df_with_gemma_12b.head())

overall_acc_gemma_4b = (vote_df_with_gemma_4b['complex'] == 0).mean()
overall_acc_gemma_12b = (vote_df_with_gemma_12b['complex'] == 0).mean()

print(f"Overall accuracy (complex=0 / total) with Gemma 4B: {overall_acc_gemma_4b:.4f}")
print(f"Overall accuracy (complex=0 / total) with Gemma 12B: {overall_acc_gemma_12b:.4f}")

cat_acc_4b = vote_df_with_gemma_4b.groupby('category')['vote_accuracy'].mean().sort_index()
cat_acc_12b = vote_df_with_gemma_12b.groupby('category')['vote_accuracy'].mean().sort_index()

plt.figure(figsize=(14, 5))
sns.barplot(x=cat_acc_4b.index, y=cat_acc_4b.values, color='#4C78A8')
plt.title('Category Vote Accuracy (Gemma 4B variant)')
plt.xlabel('Category')
plt.ylabel('Average Vote Accuracy (correct models / 5)')
plt.xticks(rotation=35, ha='right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

plt.figure(figsize=(14, 5))
sns.barplot(x=cat_acc_12b.index, y=cat_acc_12b.values, color='#1F77B4')
plt.title('Category Vote Accuracy (Gemma 12B variant)')
plt.xlabel('Category')
plt.ylabel('Average Vote Accuracy (correct models / 5)')
plt.xticks(rotation=35, ha='right')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()


# Combine the 2 barchart into one side-by-side for easier comparison
cat_acc_df = pd.DataFrame({
    'Category': cat_acc_4b.index,
    '4 models + Gemma 4B': cat_acc_4b.values,
    '4 models + Gemma 12B': cat_acc_12b.values
}).melt(id_vars='Category', var_name='Variant', value_name='Average Vote Accuracy')
plt.figure(figsize=(14, 5))
sns.barplot(x='Category', y='Average Vote Accuracy', hue='Variant', data=cat_acc_df, palette=["#29D815", '#1F77B4'])
plt.title('80% Vote - Category Accuracy Comparison')
plt.xlabel('Category')
plt.ylabel('Average Vote Accuracy (correct models / 5)')
plt.xticks(rotation=35, ha='right')
plt.ylim(0, 1)
plt.legend(title='Variant')
plt.show()

In [ ]:
# 80% vote on each five_results model (question-level)
# complex = 0 if accuracy >= 0.8, else 1

five_model_frames = {
    'Gemma 4B': gemma_4b_five,
    'Gemma 12B': gemma_12b_five,
    'Granite 8B': granite_8b_five,
    'Meta 8B': meta_8b_five,
    'Mistral 7B': mistral_7b_five,
    'Qwen 7B': qwen_7b_five,
}

five_with_complex = {}
model_80_vote_accuracy = {}

for model_name, df in five_model_frames.items():
    temp = df.copy()
    temp['complex'] = (temp['accuracy'] < 0.8).astype(int)
    five_with_complex[model_name] = temp
    model_80_vote_accuracy[model_name] = (temp['complex'] == 0).mean()

accuracy_by_model_df = (
    pd.DataFrame(
        {
            'model': list(model_80_vote_accuracy.keys()),
            'accuracy': list(model_80_vote_accuracy.values())
        }
    )
    .sort_values('accuracy', ascending=False)
    .reset_index(drop=True)
)

print('80% vote accuracy by model (complex=0 / total):')
display(accuracy_by_model_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=accuracy_by_model_df, x='model', y='accuracy', color='#1f77b4')
plt.title('Five Results: 80% Vote Accuracy by Model')
plt.xlabel('Model')
plt.ylabel('Accuracy (complex=0 / total)')
plt.ylim(0, 1)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

# Add the dataset of Qwen with complex label
qwen_7b_five_with_complex = five_with_complex['Qwen 7B']
print('Qwen 7B Five Results with Complex Label:')


In [ ]:
# 4/5 ensemble vote on five_results using per-model threshold: accuracy >= 0.8 => complex = 0

def build_five_ensemble_df(model_frames, version_name):
    base = next(iter(model_frames.values()))
    out = base[['prompt', 'category', 'correct_answer']].copy()

    vote_cols = []
    for model_name, df in model_frames.items():
        col = f"{model_name}_complex_vote"
        # Model vote rule: complex=0 if accuracy >= 0.8, else complex=1
        out[col] = (df['accuracy'] < 0.8).astype(int)
        vote_cols.append(col)

    out['models_vote_complex_0'] = (out[vote_cols] == 0).sum(axis=1)
    out['models_vote_complex_1'] = 5 - out['models_vote_complex_0']

    # Ensemble rule: complex=0 if at least 4 out of 5 models vote complex=0
    out['complex'] = (out['models_vote_complex_0'] < 4).astype(int)
    out['ensemble_version'] = version_name
    return out

five_models_with_gemma_4b = {
    'Gemma 4B': gemma_4b_five,
    'Granite 8B': granite_8b_five,
    'Meta 8B': meta_8b_five,
    'Mistral 7B': mistral_7b_five,
    'Qwen 7B': qwen_7b_five,
}

five_models_with_gemma_12b = {
    'Gemma 12B': gemma_12b_five,
    'Granite 8B': granite_8b_five,
    'Meta 8B': meta_8b_five,
    'Mistral 7B': mistral_7b_five,
    'Qwen 7B': qwen_7b_five,
}

five_vote_df_with_gemma_4b = build_five_ensemble_df(five_models_with_gemma_4b, 'with_gemma_4b')
five_vote_df_with_gemma_12b = build_five_ensemble_df(five_models_with_gemma_12b, 'with_gemma_12b')

print('five_vote_df_with_gemma_4b:')
display(five_vote_df_with_gemma_4b.head())
print('five_vote_df_with_gemma_12b:')
display(five_vote_df_with_gemma_12b.head())

overall_complex0_4b = (five_vote_df_with_gemma_4b['complex'] == 0).mean()
overall_complex0_12b = (five_vote_df_with_gemma_12b['complex'] == 0).mean()

print(f"Overall ensemble accuracy (complex=0 / total) with Gemma 4B: {overall_complex0_4b:.4f}")
print(f"Overall ensemble accuracy (complex=0 / total) with Gemma 12B: {overall_complex0_12b:.4f}")

